In [1]:
pip install selenium beautifulsoup4 webdriver-manager

  Using cached selenium-4.35.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached trio-0.30.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached certifi-2025.8.3-py3-none-any.whl.metadata (2.4 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.2.0-py3-none-any.whl.metadata (5.6 kB)
Using cached selenium-4.35.0-py3-none-any.whl (9.6 MB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cached certifi-2025.8.3-py3-none-any.whl (161 kB)
Using cached trio-0.30.0-py3-none-any.whl (499 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached typing_extension

In [11]:
!pip uninstall webdriver-manager

^C


In [12]:
!pip install undetected_chromedriver

     ---------------------------------------- 0.0/65.4 kB ? eta -:--:--
     ------ --------------------------------- 10.2/65.4 kB ? eta -:--:--
     ------ --------------------------------- 10.2/65.4 kB ? eta -:--:--
     ----------------- -------------------- 30.7/65.4 kB 217.9 kB/s eta 0:00:01
     ----------------------- -------------- 41.0/65.4 kB 245.8 kB/s eta 0:00:01
     -------------------------------------- 65.4/65.4 kB 320.8 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/176.8 kB ? eta -:--:--
   --------- ------------------------------ 41.0/176.8 kB ? eta -:--:--
   --------- ------------------------------ 41.0/176.8 kB ? eta -:--:--
   --------- ------------------------------ 41.0/176.8 kB ? eta -:--:--
   -------------------- ------------------ 92.2/176.8 kB 585.1 kB/s eta 0:00:01
   ------------------------------ ------- 143.4/176.8 kB 950.9 kB/s eta 0

In [14]:
# Import necessary libraries
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import csv
import os
import random

# Base URL of the Zazzle wedding invitations category page
base_url = 'https://www.zazzle.com/c/wedding+invitations?st=orderitemcount_year'

# Initialize CSV file to store the data
csv_filename = 'zazzle_wedding_invitations.csv'
csv_headers = ['Title', 'Description', 'Price', 'Tags', 'URL', 'View Count', 'Creation Date']

# Set up Chrome options for local execution
options = Options()
options.add_argument("--remote-allow-origins=*")
# The following arguments are often used in headless mode, but are included here for robustness
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Function to extract product data from a single page
def extract_product_data(driver):
    print("Extracting data from the page...")
    time.sleep(random.uniform(5, 8))  # Wait for dynamic content to load with a random delay

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Find all product containers
    product_containers = soup.find_all('li', class_='product-item')

    product_data = []
    if not product_containers:
        print("No product items found. The CSS class names might have changed.")
        return product_data

    for product in product_containers:
        try:
            # Extracting and cleaning the data
            title = product.find('h3', class_='_text-link_1x8e2y_41').get_text(strip=True) if product.find('h3', class_='_text-link_1x8e2y_41') else 'N/A'
            description = product.find('p', class_='_text_1x8e2y_36').get_text(strip=True) if product.find('p', class_='_text_1x8e2y_36') else 'N/A'
            price = product.find('span', class_='_price_1w07i6_10').get_text(strip=True) if product.find('span', class_='_price_1w07i6_10') else 'N/A'
            tags = product.find('div', class_='_tags-list_1x8e2y_57').get_text(strip=True) if product.find('div', class_='_tags-list_1x8e2y_57') else 'N/A'
            url_element = product.find('a', class_='_text-link_1x8e2y_41')
            product_url = 'https://www.zazzle.com' + url_element['href'] if url_element and 'href' in url_element.attrs else 'N/A'
            view_count_and_date = product.find('div', class_='_meta-data_1x8e2y_65')
            
            view_count = 'N/A'
            creation_date = 'N/A'

            if view_count_and_date:
                data_elements = view_count_and_date.find_all('span')
                if len(data_elements) >= 2:
                    view_count = data_elements[0].get_text(strip=True)
                    creation_date = data_elements[1].get_text(strip=True)

            # Append the extracted data
            product_data.append([title, description, price, tags, product_url, view_count, creation_date])

        except Exception as e:
            print(f"Error extracting data from a product item: {e}")
            continue

    return product_data

# Write data to CSV
def save_to_csv(data, filename):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(csv_headers)  # Write headers
        writer.writerows(data)  # Write all the rows
    print(f"Data saved to '{filename}'. Total products: {len(data)}")

# Main function to scrape the entire page
def scrape_all_products():
    print('Starting the Zazzle scraper...')
    driver = None
    try:
        # Use undetected_chromedriver to launch the browser
        driver = uc.Chrome(options=options)
        print("WebDriver initialized successfully. Opening the page...")

        # Navigate to the URL
        driver.get(base_url)

        # Get the initial height of the page
        last_height = driver.execute_script("return document.body.scrollHeight")
        print("Scrolling down to load all products...")
        
        # Scroll down repeatedly to load all products on the page
        while True:
            # Scroll to the bottom of the page
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            # Wait for new content to load
            time.sleep(random.uniform(2, 4))
            # Calculate new scroll height and compare with last scroll height
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        print("Finished scrolling. All products should be loaded.")

        # Extract product data after all products are loaded
        product_data = extract_product_data(driver)

        if not product_data:
            print("No product data was extracted.")
            return

        # Save the product data to a CSV file
        save_to_csv(product_data, csv_filename)
        print(f"Scraping completed. Data saved to '{csv_filename}'.")

    except Exception as e:
        print(f"An error occurred during scraping: {type(e).__name__} - {e}")

    finally:
        # Close the Selenium driver after scraping is complete
        if driver:
            driver.quit()
            print("WebDriver closed successfully.")

# Run the main function
if __name__ == '__main__':
    scrape_all_products()

Starting the Zazzle scraper...
WebDriver initialized successfully. Opening the page...
An error occurred during scraping: NoSuchWindowException - Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=140.0.7339.128)
Stacktrace:
	GetHandleVerifier [0x0xf7d2a3+66419]
	GetHandleVerifier [0x0xf7d2e4+66484]
	(No symbol) [0x0xd54bd3]
	(No symbol) [0x0xd3343d]
	(No symbol) [0x0xdc786e]
	(No symbol) [0x0xde21b9]
	(No symbol) [0x0xdc0e16]
	(No symbol) [0x0xd925ce]
	(No symbol) [0x0xd934a4]
	GetHandleVerifier [0x0x11c5ee3+2461619]
	GetHandleVerifier [0x0x11c0f66+2441270]
	GetHandleVerifier [0x0xfa6242+234258]
	GetHandleVerifier [0x0xf96208+168664]
	GetHandleVerifier [0x0xf9d1ad+197245]
	GetHandleVerifier [0x0xf855f8+100040]
	GetHandleVerifier [0x0xf85792+100450]
	GetHandleVerifier [0x0xf6f74a+10266]
	BaseThreadInitThunk [0x0x76ddfcc9+25]
	RtlGetAppContainerNamedObjectPath [0x0x76f282ae+286]
	RtlGetAppContainerNamedObjectPath [0x0x76f